# Lumen on Kaggle

The code lives in the private GitHub repo; this notebook only clones it and runs `kaggle/run_stages.py`. Details: `kaggle/README.md`.

**Settings (right sidebar):**
- **Accelerator:** *None* for Stage 1 cells, *GPU T4* for Stage 2.
- **Internet:** on. **Persistence:** *Files only* or *Variables and Files* (keeps the 15 GB of data and partial runs between sessions).
- **Secrets (Add-ons → Secrets):** `GITHUB_TOKEN` (read access to the repo) and `HF_TOKEN` (write access to `cjitendr/lumen-predictions` and `cjitendr/lumen-adapter` only).

**Every session:** run **1 → 2 → 3** first (clone/update, setup, environment). Then jump to the stage you need.

In [ ]:
# Clone or update the private repo. The GitHub token never appears in a command line, a URL,
# notebook output, an exception, or on disk: git receives it only as an HTTP auth header through
# environment variables set for the git process alone (tests/test_notebook_clone.py enforces this).
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

REPO_URL = "https://github.com/JCHETAN26/lumen.git"
WORKDIR = os.environ.get("LUMEN_KAGGLE_WORKDIR", "/kaggle/working")


def _git(*args: str) -> None:
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = {k: v for k, v in os.environ.items() if not k.startswith("GIT_TRACE")}
    env.update(
        GIT_TERMINAL_PROMPT="0",  # fail instead of prompting for credentials
        GIT_CONFIG_COUNT="1",
        GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
        GIT_CONFIG_VALUE_0=f"Authorization: Basic {basic}",
    )
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True, cwd=WORKDIR)
    message = (result.stderr or result.stdout or "").strip()
    for secret in (token, basic):  # redact, in case git ever echoes the header
        message = message.replace(secret, "***")
    del token, basic, env
    if result.returncode != 0:
        raise RuntimeError(
            f"git {args[0]} failed (exit {result.returncode}): {message[-800:]}"
        ) from None


os.makedirs(WORKDIR, exist_ok=True)
if not os.path.isdir(os.path.join(WORKDIR, "lumen")):
    _git("clone", "--depth", "1", REPO_URL, "lumen")
else:
    _git("-C", "lumen", "pull", "--ff-only", REPO_URL, "main")
head = subprocess.run(
    ["git", "-C", os.path.join(WORKDIR, "lumen"), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
).stdout.strip()
print("repo at", head)

In [ ]:
# 2. Setup: installs the pinned environment (CUDA PyTorch only when a GPU is attached). Idempotent.
%cd /kaggle/working/lumen
!bash kaggle/setup.sh

In [ ]:
# 3. Environment: records the machine, GPU, and library versions.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py env

## Stage 1 (done 2026-09-28): data and Mac sample. CPU only.

Skip these unless the data check in S2.0 says the zips are missing. Run them with **Accelerator: None**.

In [ ]:
# S1.1 Download + verify the pinned VizWiz files (~15 GB, resumable), build the splits.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py data

In [ ]:
# S1.2 Package the 300-image Mac sample (already built; prints 'exists' if so).
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py mac_sample

## Stage 2: zero-shot baselines (Accelerator: GPU T4)

Order: **S2.0 → S2.1 → S2.2 → S2.3 → S2.4 → S2.5 → S2.6**, then send the outputs for review before **S2.7** and **S2.8**.

Every GPU run checks its estimate + 1 h reserve against `GPU_HOURS_LEFT`, re-checks after a timed calibration batch, and resumes where it stopped if the session ends. Prediction files go only to the private Hugging Face dataset, never to git or the download folder.

In [ ]:
# S2.0 Data present? Expect Annotations.zip, train.zip (~11.3 GB), val.zip (~3.5 GB).
# If missing: switch to Accelerator None and run S1.1 first (downloading on GPU wastes quota).
!ls -la /kaggle/working/lumen-data/

In [ ]:
# S2.1 Quota + Hugging Face token. The token is exported for the `!` commands below and
# never printed.
import os

import kaggle_secrets

GPU_HOURS_LEFT = 30.0  # <- set to your real remaining weekly quota (kaggle.com -> Settings)
os.environ["HF_TOKEN"] = kaggle_secrets.UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# S2.2 Storage check (seconds): both private HF repos exist, are private, and HF_TOKEN can write.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py storage_check

In [ ]:
# S2.3 Plan (free): prior GPU hours per run and whether the dev runs fit. Loads nothing.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model all --split dev --prompt both --dry-run --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.4 Quick check 1 (~2-5 min): tiny random-weight stand-ins with the real processors on the
# GPU. Proves CUDA + data + pipeline. Written under baseline/stand-in/, never uploaded.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m lfm2.5-vl-450m --split dev --prompt abstain --stand-in --limit 16 --device cuda

In [ ]:
# S2.5 Quick check 2 + speed A/B (~5 min total): real SmolVLM2-500M in fp16 on fresh dev
# questions. Checks for NaN/overflow and measures s/question with and without prefetching.
# Each command answers 64 new questions (32 are already done) and prints a line
# "measured X s/question ...". Send both lines. Smoke results only, never uploaded.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --limit 96 --prefetch-workers 2 --gpu-hours-left $GPU_HOURS_LEFT
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model smolvlm2-500m --split dev --prompt abstain --limit 160 --prefetch-workers 0 --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.6 Summary so far (one line per result: accuracy, CI, AP, % unanswerable, s/question, GPU h).
# Send this output (plus S2.2) for review before S2.7.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

In [ ]:
# S2.7 Dev runs: 4 models x 2 prompts. Prompt choices are made here, on dev only.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model all --split dev --prompt both --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.8 Val runs: each model with the prompt it scored best with on dev.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline --model all --split val --prompt dev-selected --gpu-hours-left $GPU_HOURS_LEFT

In [ ]:
# S2.9 Final summary.
!source /kaggle/working/lumen-env.sh && uv run python kaggle/run_stages.py baseline_summary

## Download

Download `/kaggle/working/lumen-results/` from the **Output** tab: result JSONs (aggregate scores + metadata), split ID lists, and the Mac sample archive. Prediction files are **not** in it: completed runs upload them to the private Hugging Face dataset, and each result records their Hub revision and sha256. Image zips and local prediction copies stay on Kaggle.